
# Редактор факторов `df_out`

Этот ноутбук добавляет интерактивный редактор на базе `ipywidgets`.

Возможности:
- поиск по **УНП**, **наименованию клиента**, **номеру договора**;
- фильтры по **валюте**, **типу операции**, **отчетной дате**;
- просмотр `задолженность` и `OD`;
- редактирование `НИ`, `ПФН`, `НВВ`, `рестра`, `обеспеченность`, `ГР`, `%рез`;
- сохранение изменений обратно в `df_out`;
- журнал ручных изменений `manual_edit_log`.

> Перед запуском этого ноутбука в текущем kernel должен существовать DataFrame `df_out`.


In [ ]:

import re
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output


## Настройки названий постоянных столбцов

In [ ]:

COL_UNN = "УНП"
COL_CLIENT = "Наименование клиента"
COL_CONTRACT = "Номер договора"
COL_CURRENCY = "Валюта"
COL_OPERATION = "Тип операции"

required_columns = [
    COL_UNN,
    COL_CLIENT,
    COL_CONTRACT,
    COL_CURRENCY,
    COL_OPERATION,
]

missing_columns = [col for col in required_columns if col not in df_out.columns]

if missing_columns:
    raise ValueError(
        "В df_out отсутствуют обязательные столбцы: "
        + ", ".join(missing_columns)
    )


## Поиск отчетных дат

In [ ]:

report_dates = []

for col in df_out.columns:
    match = re.match(
        r"^задолженность_(\d{2}\.\d{2}\.\d{4})$",
        str(col)
    )

    if match:
        report_dates.append(match.group(1))

report_dates = sorted(
    set(report_dates),
    key=lambda x: pd.to_datetime(x, format="%d.%m.%Y")
)

if not report_dates:
    raise ValueError(
        "В df_out не найдены столбцы вида "
        "'задолженность_01.01.2026'"
    )

print("Найдено отчетных дат:", len(report_dates))
print(report_dates)


## Интерактивный редактор

In [ ]:

# =============================================================================
# СПИСКИ ДЛЯ ФИЛЬТРОВ
# =============================================================================

currency_options = ["Все"] + sorted(
    df_out[COL_CURRENCY]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

operation_options = ["Все"] + sorted(
    df_out[COL_OPERATION]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

def collect_factor_options(prefix):
    cols = [col for col in df_out.columns if str(col).startswith(prefix)]

    if not cols:
        return []

    values = (
        df_out[cols]
        .stack()
        .dropna()
        .astype(str)
        .str.strip()
    )

    return sorted(
        value
        for value in values.unique()
        if value.lower() not in {"", "0", "0.0", "nan", "none"}
    )

security_options = collect_factor_options("обеспеченность_")
restra_options = collect_factor_options("рестра_")


# =============================================================================
# ПОИСКОВЫЕ ПОЛЯ
# =============================================================================

search_unn = widgets.Text(
    placeholder="Введите УНП...",
    description="УНП:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "90px"},
)

search_client = widgets.Text(
    placeholder="Часть названия клиента...",
    description="Клиент:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "90px"},
)

search_contract = widgets.Text(
    placeholder="Номер договора...",
    description="Договор:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "90px"},
)


# =============================================================================
# ФИЛЬТРЫ
# =============================================================================

currency_filter = widgets.Dropdown(
    options=currency_options,
    value="Все",
    description="Валюта:",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "90px"},
)

operation_filter = widgets.Dropdown(
    options=operation_options,
    value="Все",
    description="Операция:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "90px"},
)

date_selector = widgets.Dropdown(
    options=report_dates,
    value=report_dates[0],
    description="Дата:",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "90px"},
)

reset_button = widgets.Button(
    description="Сбросить фильтры",
    icon="refresh",
    layout=widgets.Layout(width="190px"),
)


# =============================================================================
# РЕЗУЛЬТАТЫ ПОИСКА
# =============================================================================

result_selector = widgets.Select(
    options=[],
    description="Найдено:",
    rows=12,
    layout=widgets.Layout(width="1000px", height="250px"),
    style={"description_width": "90px"},
)

result_count = widgets.HTML()


# =============================================================================
# ИНФОРМАЦИЯ О ВЫБРАННОЙ СТРОКЕ
# =============================================================================

selected_info = widgets.HTML(
    value="<i>Выберите строку из результатов поиска</i>"
)

debt_widget = widgets.Text(
    description="Задолженность:",
    disabled=True,
    layout=widgets.Layout(width="400px"),
    style={"description_width": "130px"},
)

od_widget = widgets.Text(
    description="OD:",
    disabled=True,
    layout=widgets.Layout(width="400px"),
    style={"description_width": "130px"},
)


# =============================================================================
# РЕДАКТИРУЕМЫЕ ФАКТОРЫ
# =============================================================================

ni_widget = widgets.Dropdown(
    options=[0, 1],
    description="НИ:",
    layout=widgets.Layout(width="220px"),
)

pfn_widget = widgets.Dropdown(
    options=[0, 1],
    description="ПФН:",
    layout=widgets.Layout(width="220px"),
)

nvv_widget = widgets.Dropdown(
    options=[0, 1],
    description="НВВ:",
    layout=widgets.Layout(width="220px"),
)

restra_widget = widgets.Combobox(
    options=restra_options,
    placeholder="0 / значение рестры",
    description="Рестра:",
    ensure_option=False,
    layout=widgets.Layout(width="500px"),
    style={"description_width": "130px"},
)

security_widget = widgets.Combobox(
    options=security_options,
    placeholder="Введите обеспеченность...",
    description="Обеспеченность:",
    ensure_option=False,
    layout=widgets.Layout(width="600px"),
    style={"description_width": "130px"},
)

gr_widget = widgets.Dropdown(
    options=[0, 1, 2, 3, 4, 5, 6],
    description="ГР:",
    layout=widgets.Layout(width="220px"),
)

reserve_widget = widgets.FloatText(
    description="% рез:",
    value=0,
    layout=widgets.Layout(width="250px"),
)


# =============================================================================
# КНОПКИ И СТАТУС
# =============================================================================

save_button = widgets.Button(
    description="Сохранить в df_out",
    button_style="success",
    icon="save",
    disabled=True,
    layout=widgets.Layout(width="230px", height="42px"),
)

reload_button = widgets.Button(
    description="Отменить изменения",
    icon="undo",
    disabled=True,
    layout=widgets.Layout(width="210px", height="42px"),
)

status_output = widgets.Output()


# =============================================================================
# ЖУРНАЛ РУЧНЫХ ИЗМЕНЕНИЙ
# =============================================================================

if "manual_edit_log" not in globals():
    manual_edit_log = pd.DataFrame(
        columns=[
            "index",
            "УНП",
            "Номер договора",
            "дата",
            "поле",
            "старое значение",
            "новое значение",
        ]
    )


# =============================================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =============================================================================

def safe_binary(value):
    try:
        return 1 if float(value) == 1 else 0
    except:
        return 0


def safe_gr(value):
    try:
        value = int(float(value))
        if value in [0, 1, 2, 3, 4, 5, 6]:
            return value
        return 0
    except:
        return 0


def safe_float(value):
    try:
        if pd.isna(value):
            return 0.0
        return float(value)
    except:
        return 0.0


def safe_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {"nan", "none", "0", "0.0"}:
        return ""

    return value


# =============================================================================
# ФИЛЬТРАЦИЯ
# =============================================================================

def get_filtered_df():
    mask = pd.Series(True, index=df_out.index)

    unn_text = search_unn.value.strip().lower()
    if unn_text:
        mask &= (
            df_out[COL_UNN]
            .astype(str)
            .str.lower()
            .str.contains(unn_text, regex=False, na=False)
        )

    client_text = search_client.value.strip().lower()
    if client_text:
        mask &= (
            df_out[COL_CLIENT]
            .astype(str)
            .str.lower()
            .str.contains(client_text, regex=False, na=False)
        )

    contract_text = search_contract.value.strip().lower()
    if contract_text:
        mask &= (
            df_out[COL_CONTRACT]
            .astype(str)
            .str.lower()
            .str.contains(contract_text, regex=False, na=False)
        )

    if currency_filter.value != "Все":
        mask &= (
            df_out[COL_CURRENCY]
            .astype(str)
            .str.strip()
            .eq(str(currency_filter.value).strip())
        )

    if operation_filter.value != "Все":
        mask &= (
            df_out[COL_OPERATION]
            .astype(str)
            .str.strip()
            .eq(str(operation_filter.value).strip())
        )

    return df_out.loc[mask]


# =============================================================================
# ОБНОВЛЕНИЕ РЕЗУЛЬТАТОВ
# =============================================================================

def update_results(change=None):
    filtered = get_filtered_df()
    options = []

    for idx, row in filtered.head(500).iterrows():
        label = (
            f"{row.get(COL_UNN, '')} | "
            f"{row.get(COL_CLIENT, '')} | "
            f"дог. {row.get(COL_CONTRACT, '')} | "
            f"{row.get(COL_CURRENCY, '')} | "
            f"{row.get(COL_OPERATION, '')}"
        )
        options.append((label, idx))

    result_selector.options = options
    result_count.value = f"<b>Найдено строк: {len(filtered):,}</b>"

    if len(filtered) > 500:
        result_count.value += " — показаны первые 500"

    if len(options) == 0:
        save_button.disabled = True
        reload_button.disabled = True
        selected_info.value = "<i>Нет строк, соответствующих фильтрам</i>"


# =============================================================================
# ЗАГРУЗКА ВЫБРАННОЙ СТРОКИ
# =============================================================================

def load_selected_row(change=None):
    if result_selector.value is None:
        return

    idx = result_selector.value
    date = date_selector.value

    debt_col = f"задолженность_{date}"
    od_col = f"OD_{date}"
    ni_col = f"НИ_{date}"
    pfn_col = f"ПФН_{date}"
    nvv_col = f"НВВ_{date}"
    restra_col = f"рестра_{date}"
    security_col = f"обеспеченность_{date}"
    gr_col = f"ГР_{date}"
    reserve_col = f"%рез_{date}"

    required_date_columns = [
        ni_col,
        pfn_col,
        nvv_col,
        restra_col,
        security_col,
        gr_col,
        reserve_col,
    ]

    missing_date_columns = [
        col for col in required_date_columns
        if col not in df_out.columns
    ]

    if missing_date_columns:
        with status_output:
            clear_output()
            print(
                "Для выбранной даты отсутствуют столбцы:",
                ", ".join(missing_date_columns)
            )
        return

    row = df_out.loc[idx]

    selected_info.value = f"""
    <div style="
        padding: 10px;
        border: 1px solid #ccc;
        border-radius: 6px;
        margin-top: 5px;
        margin-bottom: 10px;
    ">
        <b>Выбрана строка {idx}</b><br>
        Клиент: <b>{row.get(COL_CLIENT, '')}</b><br>
        УНП: <b>{row.get(COL_UNN, '')}</b><br>
        Договор: <b>{row.get(COL_CONTRACT, '')}</b><br>
        Валюта: <b>{row.get(COL_CURRENCY, '')}</b><br>
        Тип операции: <b>{row.get(COL_OPERATION, '')}</b><br>
        Отчетная дата: <b>{date}</b>
    </div>
    """

    debt_widget.value = str(row.get(debt_col, 0))
    od_widget.value = str(row.get(od_col, 0))
    ni_widget.value = safe_binary(row.get(ni_col, 0))
    pfn_widget.value = safe_binary(row.get(pfn_col, 0))
    nvv_widget.value = safe_binary(row.get(nvv_col, 0))
    restra_widget.value = safe_text(row.get(restra_col, 0))
    security_widget.value = safe_text(row.get(security_col, 0))
    gr_widget.value = safe_gr(row.get(gr_col, 0))
    reserve_widget.value = safe_float(row.get(reserve_col, 0))

    save_button.disabled = False
    reload_button.disabled = False

    with status_output:
        clear_output()


# =============================================================================
# СОХРАНЕНИЕ ИЗМЕНЕНИЙ
# =============================================================================

def save_values(button):
    global manual_edit_log

    if result_selector.value is None:
        return

    idx = result_selector.value
    date = date_selector.value

    changes = {
        "НИ": ni_widget.value,
        "ПФН": pfn_widget.value,
        "НВВ": nvv_widget.value,
        "рестра": restra_widget.value if restra_widget.value != "" else 0,
        "обеспеченность": security_widget.value if security_widget.value != "" else 0,
        "ГР": gr_widget.value,
        "%рез": reserve_widget.value,
    }

    new_log_rows = []

    for factor, new_value in changes.items():
        column = f"{factor}_{date}"

        if column not in df_out.columns:
            continue

        old_value = df_out.at[idx, column]
        df_out.at[idx, column] = new_value

        old_normalized = "" if pd.isna(old_value) else str(old_value)
        new_normalized = str(new_value)

        if old_normalized != new_normalized:
            new_log_rows.append({
                "index": idx,
                "УНП": df_out.at[idx, COL_UNN],
                "Номер договора": df_out.at[idx, COL_CONTRACT],
                "дата": date,
                "поле": factor,
                "старое значение": old_value,
                "новое значение": new_value,
            })

    if new_log_rows:
        manual_edit_log = pd.concat(
            [manual_edit_log, pd.DataFrame(new_log_rows)],
            ignore_index=True,
        )

    with status_output:
        clear_output()
        if new_log_rows:
            print(f"✓ Сохранено изменений: {len(new_log_rows)}")
        else:
            print("Изменений не обнаружено")


# =============================================================================
# ОТМЕНА И СБРОС
# =============================================================================

def reload_values(button):
    load_selected_row()
    with status_output:
        clear_output()
        print("Значения восстановлены из df_out")


def reset_filters(button):
    search_unn.value = ""
    search_client.value = ""
    search_contract.value = ""
    currency_filter.value = "Все"
    operation_filter.value = "Все"
    update_results()


# =============================================================================
# СОБЫТИЯ
# =============================================================================

search_unn.observe(update_results, names="value")
search_client.observe(update_results, names="value")
search_contract.observe(update_results, names="value")
currency_filter.observe(update_results, names="value")
operation_filter.observe(update_results, names="value")
result_selector.observe(load_selected_row, names="value")
date_selector.observe(load_selected_row, names="value")

save_button.on_click(save_values)
reload_button.on_click(reload_values)
reset_button.on_click(reset_filters)


# =============================================================================
# ИНТЕРФЕЙС
# =============================================================================

title = widgets.HTML(
    '<h2 style="margin-bottom:5px;">Редактор факторов df_out</h2>'
)

search_box = widgets.VBox([
    widgets.HBox([search_unn, search_contract]),
    search_client,
])

filter_box = widgets.HBox([
    currency_filter,
    operation_filter,
    date_selector,
    reset_button,
])

financial_box = widgets.HBox([
    debt_widget,
    od_widget,
])

binary_factor_box = widgets.HBox([
    ni_widget,
    pfn_widget,
    nvv_widget,
])

risk_box = widgets.HBox([
    gr_widget,
    reserve_widget,
])

button_box = widgets.HBox([
    save_button,
    reload_button,
])

editor_ui = widgets.VBox([
    title,
    widgets.HTML("<h4>Поиск</h4>"),
    search_box,
    widgets.HTML("<h4>Фильтры</h4>"),
    filter_box,
    result_count,
    result_selector,
    widgets.HTML("<h4>Редактирование выбранной строки</h4>"),
    selected_info,
    financial_box,
    widgets.HTML("<b>Факторы:</b>"),
    binary_factor_box,
    restra_widget,
    security_widget,
    risk_box,
    button_box,
    status_output,
])

update_results()
display(editor_ui)


## Журнал ручных изменений

In [ ]:
display(manual_edit_log)


## Проверка результата

После сохранения через виджет изменения уже находятся непосредственно в `df_out`.


In [ ]:
df_out.head()